# Day 40 — A/B Testing Analytics\n## Experiment: Control vs Experiment Conversion Performance\n\nThis notebook analyzes a simulated A/B test, calculates statistical significance, visualizes outcomes, and converts the results into a business recommendation.

### Objective\n- Compare conversion rates between control and experiment groups.\n- Test whether the observed difference is statistically significant.\n- Estimate lift and provide a practical business recommendation.\n- Use a two-proportion z-test with α = 0.05.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.stats import norm\n\ndf = pd.read_csv('../data/ab_test_data.csv')\ndf.head()

In [ ]:
summary = df.groupby('group').agg(\n    users=('user_id','count'),\n    conversions=('converted','sum'),\n    revenue=('revenue','sum'),\n    avg_session_duration=('session_duration_sec','mean')\n)\nsummary['conversion_rate'] = summary['conversions'] / summary['users']\nsummary['revenue_per_user'] = summary['revenue'] / summary['users']\nsummary

In [ ]:
# Two-proportion z-test\nc = summary.loc['control']\ne = summary.loc['experiment']\np1, p2 = c['conversion_rate'], e['conversion_rate']\nn1, n2 = c['users'], e['users']\nx1, x2 = c['conversions'], e['conversions']\n\npooled = (x1 + x2) / (n1 + n2)\nse = np.sqrt(pooled * (1-pooled) * (1/n1 + 1/n2))\nz = (p2 - p1) / se\np_value = 2 * (1 - norm.cdf(abs(z)))\nlift = (p2 - p1) / p1\n\nprint(f'Control conversion rate: {p1:.2%}')\nprint(f'Experiment conversion rate: {p2:.2%}')\nprint(f'Absolute lift: {(p2-p1):.2%}')\nprint(f'Relative lift: {lift:.2%}')\nprint(f'z-statistic: {z:.3f}')\nprint(f'p-value: {p_value:.6f}')\nprint('Statistically significant:', p_value < 0.05)

In [ ]:
groups = ['control','experiment']\nrates = [summary.loc[g,'conversion_rate'] for g in groups]\nplt.figure(figsize=(7,5))\nbars = plt.bar(groups, rates)\nplt.ylabel('Conversion Rate')\nplt.title('A/B Test Conversion Rate')\nplt.ylim(0, max(rates)*1.25)\nfor b, r in zip(bars, rates):\n    plt.text(b.get_x()+b.get_width()/2, r, f'{r:.2%}', ha='center', va='bottom')\nplt.tight_layout()\nplt.show()

## Business Interpretation\nIf the experiment produces a statistically significant improvement, the recommended action is to roll out the experiment variant gradually while monitoring downstream metrics such as revenue per user, retention, and guardrail metrics. Statistical significance alone should not be treated as proof of long-term business value.

In [ ]:
alpha = 0.05\nif p_value < alpha and p2 > p1:\n    recommendation = 'RECOMMEND ROLLOUT: the experiment has a statistically significant positive conversion effect.'\nelif p_value < alpha and p2 < p1:\n    recommendation = 'DO NOT ROLL OUT: the experiment has a statistically significant negative conversion effect.'\nelse:\n    recommendation = 'CONTINUE TESTING: the observed difference is not statistically significant at α=0.05.'\nprint(recommendation)